# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [33]:
# Initialize and constants
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')
 
if api_key and api_key.startswith('AIz') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
   
MODEL = 'gemini-2.5-flash-lite'
openai = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

API key looks good so far


In [14]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [10]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'}]}

In [37]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash-lite
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'news page',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [19]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 14 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/blog'},
  {'type': 'company page', 'url': 'https://huggingface.co/learn'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'company page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'company page', 'url': 'https://huggingface.co/docs'},
  {'type': 'company page', 'url': 'https://huggingface.co/changelog'},
  {'type': 'company page', 'url': 'https://huggingface.co/chat'},
  {'type': 'company page', 'url': 'https://huggingface.co/models'},
  {'type': 'company page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'company page', 'url': 'https://huggingface.co/spaces'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [20]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [21]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5
Updated
6 days ago
•
172k
•
1.36k
MiniMaxAI/MiniMax-M2.5
Updated
3 days ago
•
89.9k
•
762
Qwen/Qwen3.5-397B-A17B
Updated
3 days ago
•
79.3k
•
699
Nanbeige/Nanbeige4.1-3B
Updated
about 6 hours ago
•
77.3k
•
582
nvidia/personaplex-7b-v1
Updated
4 days ago
•
479k
•
2.04k
Browse 2M+ models
Spaces
Running
768
Demo Playground
⚡
768
Free platform to access multiple AI models
Running
Reachy
302
Reachy Phone Home
📱
302
Phone focus companion for Reachy Mini
Running
Featured
4.75k
Wan2.2 Animate
👁
4.75k
Wan2.2 Animate
R

In [22]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Found 8 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\n6 days ago\n•\n172k\n•\n1.36k\nMiniMaxAI/MiniMax-M2.5\nUpdated\n3 days ago\n•\n89.9k\n•\n762\nQwen/Qwen3.5-397B-A17B\nUpdated\n3 days ago\n•\n79.3k\n•\n699\nNanbeige/Nanbeige4.1-3B\nUpdated\nabout 6 hours ago\n•\n77.3k\n•\n582\nnvidia/personaplex-7b-v1\nUpdated\n4 days ago\n•\n479k\n•\n2.04k\nBrowse 2M+ models\nSpaces\nRunning\n768\nDemo Playground\n⚡\n768\nFree plat

In [36]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-flash-latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Found 19 relevant links


# Hugging Face: The Heart of the AI Community

Hugging Face is the leading collaboration platform for the machine learning community. As the "Home of Machine Learning," we provide the infrastructure and ecosystem for the next generation of engineers, scientists, and enthusiasts to build, discover, and experiment with open-source AI.

## The Hub for Innovation

Hugging Face is more than just a repository; it is a massive ecosystem where the world’s most advanced AI research is shared and refined.

*   **Models:** Explore over 2 million open-source models covering text, image, video, audio, and 3D modalities.
*   **Datasets:** Access more than 500,000 datasets to train and evaluate your machine learning projects.
*   **Spaces:** Discover over 1 million AI applications and interactive demos where you can test the latest breakthroughs directly in your browser.
*   **Open Source Stack:** Accelerate your development with our industry-standard libraries and tools.

## Empowering Enterprise

While we are rooted in open source, Hugging Face provides the tools necessary for businesses to scale AI safely and efficiently.

*   **Enterprise Hub:** Secure, private collaboration with advanced access controls and dedicated support.
*   **Compute Solutions:** Paid compute resources to run and train models without the infrastructure headache.
*   **Security:** Enterprise-grade security protocols to protect proprietary data and models.

## Our Culture and Values

Hugging Face is built on the belief that AI should be open and ethical. Our culture is defined by:

*   **Radical Collaboration:** We believe the best AI is built when the community works together, sharing research and benchmarks.
*   **Ethics at the Core:** We are committed to fostering an ethical AI landscape, actively supporting research into fairness, transparency, and safety.
*   **Multidisciplinary Expertise:** Our community and staff work across various fields, including NLP, Computer Vision, Audio, Robotics (LeRobot), and Reinforcement Learning.

## For Prospective Customers and Partners

We work with global leaders—including NVIDIA, IBM, and UC Berkeley—to bridge the gap between research and enterprise-ready applications. Whether you are looking to host private models or leverage the latest in open-source innovation, Hugging Face provides the stability and support your organization needs.

## Join the Future of AI

Hugging Face is looking for individuals who are passionate about democratizing machine learning. We offer a unique environment where you can:

*   **Build Your Portfolio:** Share your work with millions of users and establish your profile in the ML world.
*   **Work on the Cutting Edge:** Engage with the latest trends in Agentic RL, Large Language Models (LLMs), and Embodied AI.
*   **Impact the World:** Help build the open-source tools that are used by virtually every major AI lab and technology company today.

Whether you are a researcher, a software engineer, or an AI advocate, Hugging Face offers a place to grow and make a tangible impact on the future of technology.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [43]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [44]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the central hub of the machine learning world. As the definitive platform for AI collaboration, we empower data scientists, researchers, and ML engineers to create, discover, and host the building blocks of modern artificial intelligence.

## The Home of Machine Learning

Hugging Face is where the global AI community comes together to collaborate on the future of technology. Our platform serves as the primary repository for the open-source stack that powers the modern AI revolution.

### A Massive Open Ecosystem
*   **2 Million+ Models:** From Large Language Models (LLMs) to specialized computer vision and audio tools.
*   **500,000+ Datasets:** The raw materials needed to train and fine-tune the next generation of AI.
*   **1 Million+ Applications:** Through Hugging Face **Spaces**, users can explore and deploy live AI demos ranging from video generation to music synthesis.

## What We Offer

### For Developers and Researchers
Hugging Face provides an open-source stack that simplifies the ML lifecycle. Whether you are working with Text, Image, Video, Audio, or 3D modalities, our platform allows you to:
*   **Host & Collaborate:** Share unlimited public models and datasets.
*   **Build Your Portfolio:** Showcase your work to the world and build a professional profile within the ML community.
*   **Interoperability:** Seamlessly integrate with popular libraries like PyTorch, TensorFlow, JAX, and Transformers.

### For Teams and Enterprises
We provide the infrastructure and security required for professional-grade AI development.
*   **Enterprise-Grade Security:** Advanced access controls and security features to protect proprietary data.
*   **Dedicated Support:** Expert guidance to help teams accelerate their ML roadmaps.
*   **Paid Compute Solutions:** Scalable compute resources to train and deploy models efficiently.

## Our Culture and Community

At its heart, Hugging Face is driven by the spirit of **Open Source**. We believe that AI should be accessible, transparent, and collaborative. Our culture is one of rapid innovation where we move fast to support every new modality and breakthrough in the field. 

We foster an environment where developers don't just use tools—they build them together. By democratizing access to state-of-the-art AI, we enable a diverse global community to contribute to the most important technology of our time.

## Careers at Hugging Face

Hugging Face is looking for passionate individuals who want to build the "GitHub of Machine Learning." While we are a platform for code and models, we are ultimately a company built on community. 

**Why Join Us?**
*   **Impact:** Work at the epicenter of the AI industry.
*   **Community First:** Engage with millions of users and the world’s leading AI researchers.
*   **Innovation:** Help build the tools that are used by companies like NVIDIA, Google, and thousands of startups to define the future of technology.

*Join the community and help us build the future of AI.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>